# Build Consensus-Driven Multi-Agent AI Systems with Debate Pattern

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_debate_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build a debate/ensemble agent system where multiple agents solve the same task, critique each other, and reach consensus.

**Pattern:** Parallel initial responses → Iterative debate rounds → Synthesis

```
Task: "Calculate 5 factorial and explain"
  ↓
Round 0 (Parallel):
  Agent A: "5! = 120 (5×4×3×2×1)"
  Agent B: "5! equals 120"
  Agent C: "Code: factorial(5) = 120"
  ↓
Round 1 (Debate):
  Agent A: "Agent C's code is good, but B needs more detail" → Refines
  Agent B: "A is right, adding explanation..." → Refines
  Agent C: "My code works but should add comments" → Refines
  ↓
Synthesis:
  Judge combines best parts → "5! = 120 (5×4×3×2×1). Code: factorial(5)..."
```

**Key concepts:** Diverse perspectives, peer critique, iterative refinement, consensus building.

**Why Debate?**
- ✅ **Accuracy**: Multiple solvers catch errors
- ✅ **Robustness**: Reduces individual agent failures
- ✅ **Diverse perspectives**: Different approaches surface alternatives
- ✅ **Self-correction**: Agents improve by critiquing each other
- ✅ **Quality**: Synthesis produces better results than any single agent

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [ ]:
view_file("requirements.txt")

In [ ]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [ ]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the debate workflow. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [ ]:
!python -m workflows.debate --local --request "Calculate 5 factorial and explain what it means" --agents math,math,code --rounds 2

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [ ]:
!python -m workflows.debate --request "Calculate 5 factorial and explain what it means" --agents math,math,code --rounds 2

# Code Walkthrough

Let's walk through the code to understand how the debate workflow is structured and how it works.

**Note:** Debate uses the same agents and tools as all other workflows. The key difference is the orchestration pattern - multiple agents solving the same task with peer critique.

## Infrastructure

Debate uses the same infrastructure as other workflows:

- **Decorators** (`utils/decorators.py`) - Registration for agents and tools
- **Plan Executor** (`utils/plan_executor.py`) - Executes LLM-generated tool plans
- **Agents** (`agents/`) - Same specialist agents (math, string, web_search, code, weather)
- **Tools** (`tools/`) - Same toolsets for each agent

See the [planner tutorial](tutorial_planner_agent.ipynb) for details on these components.

## Debate Orchestrator - Consensus Through Critique

The debate pattern uses multiple agents solving the same task with iterative peer review.

**Flow:**
1. **Round 0 - Initial Responses:** All agents solve task independently in parallel
   - Ensures diverse thinking (no groupthink)
   - Each agent uses their own approach

2. **Rounds 1-N - Debate:**
   - Each agent sees all other responses
   - Critiques others' approaches
   - Refines their own response
   - Rates confidence (1-10)

3. **Final Synthesis:**
   - **Judge method:** LLM combines best parts from all responses
   - **Vote method:** Highest confidence response wins

**Configuring Participants:**
```python
# Same agent multiple times (diversity through randomness/temperature)
agent_names = ["math", "math", "math"]

# Mixed agent types (different perspectives)
agent_names = ["math", "code", "string"]

# Custom combinations
agent_names = ["web_search", "web_search", "code"]
```

**Debate Round Structure:**
```python
debate_prompt = f"""
Task: {user_task}

All agents' responses:
{responses_from_all_agents}

Your previous response:
{your_response}

1. Critique others' responses
2. Refine your own response
3. Rate your confidence (1-10)
"""
```

**Key features:**
- Parallel initial thinking prevents bias
- Confidence scores guide synthesis
- Iterative refinement improves quality
- Multiple synthesis methods (judge vs vote)

**Agent routing:** Uses `agent_registry` for dynamic dispatch (same as all workflows).

In [ ]:
view_file("workflows/debate.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.debate --local \
  --request "your task" \
  --agents math,math,code \
  --rounds 2 \
  --synthesis judge
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.debate \
  --request "your task" \
  --agents math,code,string \
  --rounds 2 \
  --synthesis vote
```
Distributed Flyte cluster, scalable and observable.

**Parameters:**
- `--agents`: Comma-separated list (e.g., `math,math,code` or `web_search,code`)
- `--rounds`: Number of debate rounds (default: 2)
- `--synthesis`: `judge` (LLM combines) or `vote` (highest confidence wins)

**Try these:**

In [ ]:
# Three math agents debate (diversity through randomness)
!python -m workflows.debate --local --request "Calculate 10 factorial" --agents math,math,math --rounds 2 --synthesis judge

In [ ]:
# Mixed agents for different perspectives
!python -m workflows.debate --local --request "What is 120?" --agents math,code,string --rounds 1 --synthesis vote

In [ ]:
# Complex task with 3 rounds
!python -m workflows.debate --local --request "Explain prime numbers and give examples" --agents math,code --rounds 3 --synthesis judge

---

## Key Takeaways

**Debate Pattern:**
- **Diverse perspectives:** Multiple solvers surface different approaches
- **Self-correction:** Agents improve by critiquing peers
- **Quality through consensus:** Final answer better than any individual
- **Confidence scoring:** Guides synthesis and identifies uncertainty

**When to use Debate:**

| Use Debate When | Don't Use Debate When |
|-----------------|----------------------|
| High-stakes decisions | Simple, deterministic tasks |
| Need diverse perspectives | Speed is critical |
| Complex reasoning tasks | Single clear answer exists |
| Want to reduce errors | Cost is primary concern |
| Fact-checking important | Task has unique solution path |

**Comparison with other patterns:**

| Pattern | Agents | Strength | Weakness |
|---------|--------|----------|----------|
| **Debate** | Multiple on same task | Accuracy & robustness | More LLM calls, slower |
| **Planner** | Multiple on different tasks | Parallelism | Fixed plan |
| **ReAct** | Single adaptive | Flexibility | Sequential only |
| **Reflection** | Single iterative | Quality | Slower |
| **Sequential** | Predefined pipeline | Predictable | No adaptation |

**Synthesis methods:**
- **Judge:** Better for complex tasks where combining insights is valuable
- **Vote:** Simpler, faster, good when one clear best answer emerges

**Real-world applications:**
- **Medical diagnosis:** Multiple specialists review same case
- **Code review:** Different reviewers catch different issues
- **Research synthesis:** Combine perspectives on complex topics
- **Decision making:** Board of advisors pattern
- **Quality assurance:** Ensemble classifiers for higher accuracy

**Architecture benefits:**
- Same agents/tools work with all patterns
- Type-safe, observable, scalable with Flyte
- Mix patterns (e.g., planner delegates to debate for important steps)

**Next steps:**
1. Experiment with different agent combinations
2. Try different numbers of debate rounds
3. Compare judge vs vote synthesis on same task
4. Combine patterns: use debate for high-stakes steps in larger workflows

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Planner tutorial: [tutorial_planner_agent.ipynb](tutorial_planner_agent.ipynb)
- ReAct tutorial: [tutorial_react_agent.ipynb](tutorial_react_agent.ipynb)
- Reflection tutorial: [tutorial_reflection_agent.ipynb](tutorial_reflection_agent.ipynb)
- Sequential tutorial: [tutorial_sequential_agent.ipynb](tutorial_sequential_agent.ipynb)
- Flyte docs: https://docs.flyte.org
- Research: ["Improving Factuality and Reasoning in Language Models through Multiagent Debate"](https://arxiv.org/abs/2305.14325)
- Questions? Join the Flyte community Slack!